# Netflix 电影与电视节目数据分析

---

## 小组分工（两人组）

- **成员 A（数据科学家）**：负责数据加载、数据预处理（清洗、缺失值处理、异常值处理）、描述性统计与分组统计分析，编写核心数据分析代码与可视化方案设计。
- **成员 B（前端工程师）**：负责基于 pyecharts 的交互式图表开发，搭建网页前端 `index.html`，整合所有可视化图表与结论，负责最终汇报的页面呈现与答辩演示。

---

## 1. 项目需求分析

### 1.1 数据集背景
- **来源**：Netflix 官方内容目录汇编
- **规模**：8,809 条记录，12 列
- **关键字段**：`type`（类型）、`title`（标题）、`director`（导演）、`cast`（演员）、`country`（国家）、`date_added`（添加日期）、`release_year`（发行年份）、`rating`（分级）、`duration`（时长）、`listed_in`（流派）、`description`（描述）

### 1.2 分析目标
1. 电影与电视节目的分布对比
2. 内容随时间的增长趋势
3. 主要制片国家分布
4. 热门流派 Top10
5. 年龄评级分布
6. 电影时长分布 / 电视节目季数分布
7. 年份 × 类型交叉热力图

---

## 2. 数据加载与预处理

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from collections import Counter
from pyecharts import options as opts
from pyecharts.charts import (
    Bar, Pie, Line, Scatter, WordCloud, HeatMap, Boxplot, Funnel, Gauge, Timeline, Grid, Page
)
from pyecharts.globals import ThemeType, CurrentConfig, RenderType

CurrentConfig.ONLINE_HOST = "https://cdn.jsdelivr.net/npm/echarts@5.4.3/dist/"

In [ ]:
df = pd.read_csv('netflix_titles.csv')
print(f"原始数据形状：{df.shape}")
print(f"各列数据类型：\n{df.dtypes}")
df.head()

In [ ]:
# 缺失值统计
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'缺失数量': missing, '缺失比例(%)': missing_pct})
missing_df[missing_df['缺失数量'] > 0]

In [ ]:
# 预处理：缺失值填充 + 字段清洗
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['date_added'] = df['date_added'].fillna(df['date_added'].mode()[0])
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])
df['duration'] = df['duration'].fillna(df['duration'].mode()[0])

# 提取添加年份
df['year_added'] = df['date_added'].str.extract(r'(\d{4})').astype(int)

# 拆分多值字段（流派、国家、演员）
def split_field(s):
    if pd.isna(s) or s == 'Unknown':
        return []
    return [x.strip() for x in str(s).split(',') if x.strip()]

df['genres_list'] = df['listed_in'].apply(split_field)
df['countries_list'] = df['country'].apply(split_field)
df['cast_list'] = df['cast'].apply(split_field)

# 数值化时长：电影=分钟数，电视节目=季数
df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float).astype(int)

print(f"处理后数据形状：{df.shape}")
print(f"缺失值处理后缺失总数：{df.isnull().sum().sum()}")
df

---

## 3. 描述性统计分析

In [ ]:
# 类型分布
type_dist = df['type'].value_counts().reset_index()
type_dist.columns = ['类型', '数量']
type_dist['占比(%)'] = (type_dist['数量'] / type_dist['数量'].sum() * 100).round(2)
type_dist

In [ ]:
# 按年份添加内容数量
year_added = df.groupby(['year_added', 'type']).size().unstack().fillna(0).astype(int)
year_added = year_added.sort_index()
year_added['总计'] = year_added.sum(axis=1)
year_added

In [ ]:
# 制片国家 Top15
all_countries = []
for c in df['countries_list']:
    all_countries.extend(c)
country_count = pd.DataFrame(Counter(all_countries).most_common(15), columns=['国家', '数量'])
country_count

In [ ]:
# 热门流派 Top10
all_genres = []
for g in df['genres_list']:
    all_genres.extend(g)
genre_count = pd.DataFrame(Counter(all_genres).most_common(10), columns=['流派', '数量'])
genre_count

In [ ]:
# 评级分布
rating_dist = df['rating'].value_counts().head(10).reset_index()
rating_dist.columns = ['评级', '数量']
rating_dist

---

## 4. 使用 pyecharts 绘制高级可视化图表

In [ ]:
# 图1：电影 vs 电视节目 环形图
def chart_type_pie():
    data = [list(z) for z in zip(type_dist['类型'].tolist(), type_dist['数量'].tolist())]
    c = (
        Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="500px"))
        .add(
            series_name="数量分布",
            data_pair=data,
            radius=["40%", "70%"],
            center=["50%", "50%"],
            label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)", font_size=14, color="#fff"),
            itemstyle_opts=opts.ItemStyleOpts(border_color="#0e0e0e", border_width=2),
        )
        .set_colors(["#E50914", "#831010"])
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="🎬 Netflix 内容类型分布",
                subtitle=f"共 {df.shape[0]} 部作品",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                subtitle_textstyle_opts=opts.TextStyleOpts(color="#ccc", font_size=14),
                pos_left="center",
            ),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="middle", pos_right="10px", textstyle_opts=opts.TextStyleOpts(color="#fff")),
            toolbox_opts=opts.ToolboxOpts(is_show=False),
        )
        .set_series_opts(
            tooltip_opts=opts.TooltipOpts(trigger="item", formatter="{b}: {c} ({d}%)")
        )
    )
    return c

c1 = chart_type_pie()
c1.render('charts/chart1_type_pie.html')
c1

In [ ]:
# 图2：年份添加趋势图
def chart_year_line():
    years = [int(y) for y in year_added.index.tolist()]
    movies = year_added.get('Movie', pd.Series(0, index=year_added.index)).tolist()
    shows = year_added.get('TV Show', pd.Series(0, index=year_added.index)).tolist()
    
    c = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="550px"))
        .add_xaxis(years)
        .add_yaxis(
            "电影", movies, is_smooth=True, symbol_size=8,
            linestyle_opts=opts.LineStyleOpts(width=4, color="#E50914"),
            itemstyle_opts=opts.ItemStyleOpts(color="#E50914"),
            areastyle_opts=opts.AreaStyleOpts(opacity=0.2, color="#E50914"),
            label_opts=opts.LabelOpts(is_show=False),
        )
        .add_yaxis(
            "电视节目", shows, is_smooth=True, symbol_size=8,
            linestyle_opts=opts.LineStyleOpts(width=4, color="#FFD700"),
            itemstyle_opts=opts.ItemStyleOpts(color="#FFD700"),
            areastyle_opts=opts.AreaStyleOpts(opacity=0.2, color="#FFD700"),
            label_opts=opts.LabelOpts(is_show=False),
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="📈 Netflix 每年新增内容趋势",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                pos_left="center",
            ),
            xaxis_opts=opts.AxisOpts(
                type_="category",
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#ccc"),
            ),
            yaxis_opts=opts.AxisOpts(
                type_="value", name="数量",
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#ccc"),
                splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(opacity=0.15)),
            ),
            legend_opts=opts.LegendOpts(textstyle_opts=opts.TextStyleOpts(color="#fff"), pos_top="50px"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
        )
    )
    return c

c2 = chart_year_line()
c2.render('charts/chart2_year_line.html')
c2

In [ ]:
# 图3：制片国家 Top15 横向柱状图
def chart_country_bar():
    countries = country_count['国家'].tolist()[::-1]
    counts = country_count['数量'].tolist()[::-1]
    
    c = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="600px"))
        .add_xaxis(countries)
        .add_yaxis(
            "作品数量", counts,
            category_gap="40%",
            label_opts=opts.LabelOpts(position="right", color="#fff", font_size=12, font_weight="bold"),
            itemstyle_opts=opts.ItemStyleOpts(
                color={
                    "type": "linear",
                    "x": 0, "y": 0, "x2": 1, "y2": 0,
                    "colorStops": [[0, {"r": 229, "g": 9, "b": 20}], [1, {"r": 255, "g": 100, "b": 100}]],
                },
                border_radius=[0, 8, 8, 0],
            ),
        )
        .reversal_axis()
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="🌍 制片国家 Top 15",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                pos_left="center",
            ),
            xaxis_opts=opts.AxisOpts(
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#ccc"),
                splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(opacity=0.15)),
            ),
            yaxis_opts=opts.AxisOpts(
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#fff", font_size=13),
            ),
            legend_opts=opts.LegendOpts(is_show=False),
        )
    )
    return c

c3 = chart_country_bar()
c3.render('charts/chart3_country_bar.html')
c3

In [ ]:
# 图4：热门流派 Top10 柱状图
def chart_genre_bar():
    genres = genre_count['流派'].tolist()[::-1]
    counts = genre_count['数量'].tolist()[::-1]
    
    c = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="550px"))
        .add_xaxis(genres)
        .add_yaxis(
            "作品数量", counts,
            category_gap="45%",
            label_opts=opts.LabelOpts(position="right", color="#fff", font_size=12, font_weight="bold"),
            itemstyle_opts=opts.ItemStyleOpts(
                color={
                    "type": "linear",
                    "x": 0, "y": 0, "x2": 1, "y2": 0,
                    "colorStops": [[0, {"r": 64, "g": 158, "b": 255}], [1, {"r": 255, "g": 215, "b": 0}]],
                },
                border_radius=[0, 8, 8, 0],
            ),
        )
        .reversal_axis()
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="🎭 热门流派 Top 10",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                pos_left="center",
            ),
            xaxis_opts=opts.AxisOpts(
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#ccc"),
                splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(opacity=0.15)),
            ),
            yaxis_opts=opts.AxisOpts(
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#fff", font_size=12),
            ),
            legend_opts=opts.LegendOpts(is_show=False),
        )
    )
    return c

c4 = chart_genre_bar()
c4.render('charts/chart4_genre_bar.html')
c4

In [ ]:
# 图5：评级分布漏斗图
def chart_rating_funnel():
    data = [list(z) for z in zip(rating_dist['评级'].tolist(), rating_dist['数量'].tolist())]
    c = (
        Funnel(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="550px"))
        .add(
            series_name="评级分布",
            data_pair=data,
            label_opts=opts.LabelOpts(position="inside", formatter="{b}: {c}", color="#fff", font_size=13, font_weight="bold"),
            sort_="descending",
            gap=2,
        )
        .set_colors(["#E50914", "#FF6B6B", "#FFD700", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7", "#DDA0DD", "#A29BFE", "#81ECEC"])
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="🔖 年龄评级分布（漏斗图）",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                pos_left="center",
            ),
            legend_opts=opts.LegendOpts(textstyle_opts=opts.TextStyleOpts(color="#fff"), pos_top="50px"),
            tooltip_opts=opts.TooltipOpts(trigger="item", formatter="{b}: {c} ({d}%)"),
        )
    )
    return c

c5 = chart_rating_funnel()
c5.render('charts/chart5_rating_funnel.html')
c5

In [ ]:
# 图6：电影时长分布直方图
def chart_duration_scatter():
    movies = df[df['type'] == 'Movie']['duration_num'].tolist()
    # 生成分箱
    bins = list(range(0, int(max(movies)) + 20, 10))
    labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]
    counts, _ = np.histogram(movies, bins=bins)
    
    c = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="550px"))
        .add_xaxis(labels)
        .add_yaxis(
            "电影数量", counts.tolist(),
            category_gap="5%",
            label_opts=opts.LabelOpts(is_show=False),
            itemstyle_opts=opts.ItemStyleOpts(
                color={
                    "type": "linear",
                    "x": 0, "y": 0, "x2": 0, "y2": 1,
                    "colorStops": [[0, {"r": 255, "g": 107, "b": 107}], [1, {"r": 229, "g": 9, "b": 20}]],
                },
                border_radius=[6, 6, 0, 0],
            ),
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="⏱ 电影时长分布（分钟）",
                subtitle=f"平均时长: {int(np.mean(movies))} 分钟 | 中位数: {int(np.median(movies))} 分钟",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                subtitle_textstyle_opts=opts.TextStyleOpts(color="#ccc", font_size=13),
                pos_left="center",
            ),
            xaxis_opts=opts.AxisOpts(
                name="时长区间（分钟）",
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#ccc", rotate=45, font_size=10),
            ),
            yaxis_opts=opts.AxisOpts(
                name="电影数量",
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#ccc"),
                splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(opacity=0.15)),
            ),
            legend_opts=opts.LegendOpts(is_show=False),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="shadow"),
        )
    )
    return c

c6 = chart_duration_scatter()
c6.render('charts/chart6_duration.html')
c6

In [ ]:
# 图7：流派 × 类型 玫瑰图
def chart_genre_rose():
    # 对每个流派统计电影和 TV Show 数量
    top_genres = genre_count['流派'].head(8).tolist()
    genre_type = []
    for genre in top_genres:
        movie_count = df[df['genres_list'].apply(lambda x: genre in x) & (df['type'] == 'Movie')].shape[0]
        show_count = df[df['genres_list'].apply(lambda x: genre in x) & (df['type'] == 'TV Show')].shape[0]
        genre_type.append({'流派': genre, '电影': movie_count, '电视节目': show_count})
    
    gt_df = pd.DataFrame(genre_type)
    
    c = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="550px"))
        .add_xaxis(gt_df['流派'].tolist())
        .add_yaxis(
            "电影", gt_df['电影'].tolist(), stack="stack1",
            label_opts=opts.LabelOpts(is_show=True, color="#fff", position="inside"),
            itemstyle_opts=opts.ItemStyleOpts(color="#E50914", border_radius=[4, 4, 0, 0]),
        )
        .add_yaxis(
            "电视节目", gt_df['电视节目'].tolist(), stack="stack1",
            label_opts=opts.LabelOpts(is_show=True, color="#fff", position="inside"),
            itemstyle_opts=opts.ItemStyleOpts(color="#4ECDC4", border_radius=[4, 4, 0, 0]),
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="🎨 流派 × 类型 堆叠分布",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                pos_left="center",
            ),
            xaxis_opts=opts.AxisOpts(
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#fff", rotate=25, font_size=11),
            ),
            yaxis_opts=opts.AxisOpts(
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#ccc"),
                splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(opacity=0.15)),
            ),
            legend_opts=opts.LegendOpts(textstyle_opts=opts.TextStyleOpts(color="#fff"), pos_top="50px"),
        )
    )
    return c

c7 = chart_genre_rose()
c7.render('charts/chart7_genre_type.html')
c7

In [ ]:
# 图8：流派词云图
def chart_genre_cloud():
    words = [[g, int(c)] for g, c in Counter(all_genres).most_common()]
    c = (
        WordCloud(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="550px"))
        .add(
            "", words,
            word_size_range=[16, 80],
            shape="cardioid",
            textstyle_opts=opts.TextStyleOpts(font_family="Microsoft YaHei"),
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="☁️ Netflix 流派词云",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                pos_left="center",
            ),
            tooltip_opts=opts.TooltipOpts(trigger="item"),
        )
    )
    return c

c8 = chart_genre_cloud()
c8.render('charts/chart8_genre_wordcloud.html')
c8

In [ ]:
# 图9：电影 / 电视节目 季数分布
def chart_season_pie():
    shows = df[df['type'] == 'TV Show']['duration_num'].value_counts().sort_index().head(8).reset_index()
    shows.columns = ['季数', '数量']
    shows['label'] = shows['季数'].astype(str) + ' Season'
    data = [list(z) for z in zip(shows['label'].tolist(), shows['数量'].tolist())]
    
    c = (
        Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="550px"))
        .add(
            "", data,
            radius=["30%", "75%"],
            rosetype="radius",
            label_opts=opts.LabelOpts(formatter="{b}: {c}", color="#fff", font_size=12),
        )
        .set_colors(["#E50914", "#FF6B6B", "#FFD700", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7", "#DDA0DD"])
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="🌸 电视节目季数分布（玫瑰图）",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                pos_left="center",
            ),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="middle", pos_right="5%", textstyle_opts=opts.TextStyleOpts(color="#fff")),
        )
    )
    return c

c9 = chart_season_pie()
c9.render('charts/chart9_season_pie.html')
c9

In [ ]:
# 图10：年份 × 类型 热力图
def chart_heatmap():
    # 选取 2010-2021 年
    filtered = df[(df['release_year'] >= 2008) & (df['release_year'] <= 2021)]
    top_rating = rating_dist['评级'].head(8).tolist()
    r = filtered[filtered['rating'].isin(top_rating)]
    years = sorted(r['release_year'].unique().tolist())
    
    heatmap_data = []
    for y_idx, y in enumerate(years):
        for rt_idx, rt in enumerate(top_rating):
            cnt = r[(r['release_year'] == y) & (r['rating'] == rt)].shape[0]
            heatmap_data.append([rt_idx, y_idx, cnt])
    
    c = (
        HeatMap(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="600px"))
        .add_xaxis(years)
        .add_yaxis(
            "作品数量", top_rating, heatmap_data,
            label_opts=opts.LabelOpts(is_show=True, color="#fff", font_size=10),
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="🔥 发行年份 × 评级 热力图",
                title_textstyle_opts=opts.TextStyleOpts(color="#fff", font_size=22, font_weight="bold"),
                pos_left="center",
            ),
            xaxis_opts=opts.AxisOpts(
                type_="category",
                splitarea_opts=opts.SplitAreaOpts(is_show=True, areastyle_opts=opts.AreaStyleOpts(opacity=1)),
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#ccc", rotate=45),
            ),
            yaxis_opts=opts.AxisOpts(
                type_="category",
                splitarea_opts=opts.SplitAreaOpts(is_show=True, areastyle_opts=opts.AreaStyleOpts(opacity=1)),
                axisline_opts=opts.AxisLineOpts(linestyle_opts=opts.LineStyleOpts(color="#666")),
                axislabel_opts=opts.LabelOpts(color="#fff"),
            ),
            visualmap_opts=opts.VisualMapOpts(
                min_=0,
                max_=max([d[2] for d in heatmap_data]),
                range_color=["#0d0d0d", "#831010", "#E50914", "#FFD700"],
                textstyle_opts=opts.TextStyleOpts(color="#fff"),
                pos_right="20px",
                pos_bottom="20px",
            ),
            tooltip_opts=opts.TooltipOpts(trigger="item", formatter="{c} 部"),
        )
    )
    return c

c10 = chart_heatmap()
c10.render('charts/chart10_heatmap.html')
c10

---

## 5. 汇总页面（Page）

In [ ]:
# 生成汇总 DashBoard
page = Page(layout=Page.DraggablePageLayout, page_title="Netflix 数据分析看板")
for chart in [c1, c2, c3, c4, c5, c6, c7, c8, c9, c10]:
    page.add(chart)
page.render('charts/dashboard.html')
print('Dashboard 生成完成 ✓')

---

## 6. 导出 JSON 数据供前端使用

In [ ]:
os.makedirs('data', exist_ok=True)

# 1) 类型分布
type_json = type_dist.to_dict(orient='records')

# 2) 年份添加趋势
year_data = []
for idx, row in year_added.iterrows():
    year_data.append({
        'year': int(idx),
        'Movie': int(row.get('Movie', 0)),
        'TV Show': int(row.get('TV Show', 0)),
        'total': int(row['总计']),
    })

# 3) 国家 Top15
country_json = country_count.to_dict(orient='records')

# 4) 流派 Top10
genre_json = genre_count.to_dict(orient='records')

# 5) 评级
rating_json = rating_dist.to_dict(orient='records')

# 6) 电影时长统计
movie_dur = df[df['type'] == 'Movie']['duration_num'].tolist()
dur_stats = {
    'mean': float(np.mean(movie_dur)),
    'median': float(np.median(movie_dur)),
    'min': float(np.min(movie_dur)),
    'max': float(np.max(movie_dur)),
    'count': len(movie_dur),
}

# 7) 总体概览
overview = {
    'total': int(df.shape[0]),
    'movies': int(df[df['type'] == 'Movie'].shape[0]),
    'tvshows': int(df[df['type'] == 'TV Show'].shape[0]),
    'countries': len(set(c for cs in df['countries_list'] for c in cs)),
    'genres': len(set(g for gs in df['genres_list'] for g in gs)),
    'directors': int(df[df['director'] != 'Unknown']['director'].nunique()),
}

export = {
    'overview': overview,
    'type_dist': type_json,
    'year_trend': year_data,
    'countries': country_json,
    'genres': genre_json,
    'ratings': rating_json,
    'duration_stats': dur_stats,
}

with open('data/analysis_data.json', 'w', encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print('数据导出完成 ✓')
pd.DataFrame([overview])

---

## 7. 案例分析结论

### 核心发现
1. **电影占主导**：电影约占总量的 ~70%，电视节目占 ~30%，Netflix 的内容储备仍以电影为主。
2. **增长期集中在 2016-2020**：每年新增内容从 2015 年开始爆发式增长，2018-2020 年达到峰值，反映了 Netflix 的全球化扩张战略。
3. **美国是绝对核心制片国**：美国制片数量遥遥领先，印度、英国紧随其后，体现了 Netflix 对北美与南亚市场的重视。
4. **国际电影 & 剧情片最热门**：International Movies、Dramas、Comedies 是三大主流流派。
5. **电视节目多为单季剧**：约 60%+ 的 TV Show 只有 1 季，说明 Netflix 的剧集策略偏向新剧尝试。
6. **电影平均时长约 100 分钟**：符合主流商业片时长区间。

### 建议
- **内容多元化**：加强对非英语地区（如亚洲、欧洲）原创内容的投资，以匹配全球订阅用户的文化需求。
- **剧集投资**：可重点打造多季爆款 IP，以提升用户订阅粘性与留存率。
- **分级策略**：TV-MA 分级内容占比较高，可加强对家庭用户友好内容的供给，扩展受众面。